In [ ]:
#-----Agentic Graph Reasoning--- An AI agent uses tools to explore a graph, gather evidence, check relationships, 
# identify missing information, and produce a result.Unlike a conventional graph algorithm that follows a fixed procedure, 
# an agent can select which tool to call next based on what it finds. For safety and reliability, its tools should have 
# clearly defined permissions.

"""
Suppose a researcher asks:
Reconstruct this patient's oncology timeline and identify missing evidence.

The agent needs to:
1. Find the diagnosis event.
2. Traverse the connected clinical events.
3. Follow source-document references.
4. Check whether important events have supporting evidence.
5. Report missing information without inventing it.

Python program - This example uses a deterministic tool-based agent. It does not require an LLM or an external API, so you can 
see exactly how the reasoning workflow operates. This is a simple, rule-driven agent rather than an LLM-powered agent. 
A more advanced version could use an LLM to interpret a researcher's request, select tools dynamically, reconcile conflicting 
documents, and ask for clarification when evidence is insufficient.
            Block 1 — Store the clinical graph
            Events, connections and source references

            Block 2 — Create investigation tools
            Find events, follow connections and validate sources

            Block 3 — Agent investigates
            Selects the next event to inspect, tracks visited events and gathers findings

            Block 4 — Identify evidence gaps
            Flags events with missing sources or unknown events

            Block 5 — Produce the report
            Display the reconstructed timeline and evidence issues
"""
#-----Block 1 — Define the oncology patient-history graph
clinical_graph = {
    "Diagnosis": {
        "next": ["Biopsy"],
        "source": "Pathology_Report_001"
    },
    "Biopsy": {
        "next": ["Biomarker_Test"],
        "source": "Biopsy_Report_001"
    },
    "Biomarker_Test": {
        "next": ["Treatment"],
        "source": "Molecular_Report_001"
    },
    "Treatment": {
        "next": ["Response_Assessment"],
        "source": "Treatment_Record_001"
    },
    "Response_Assessment": {
        "next": [],
        "source": None                          #---The final event exists in the graph, but its source is missing.That is the problem the agent is designed to detect.
    }
}
#----Block 2 — Create the tools the agent can use . We are using 3 tools as follows 
# Tool 1: Find an event.
def find_event(event_name):
    if event_name in clinical_graph:
        return clinical_graph[event_name]
    return None
# Tool 2: Follow graph connections.
def get_next_events(event_name):
    event = find_event(event_name)
    if event is None:
        return []
    return event["next"]
# Tool 3: Check whether an event has source evidence.
def validate_source(event_name):
    event = find_event(event_name)
    if event is None:
        return False
    return bool(event["source"])
#---------- Block 3 — The agent investigates the graph
def investigate_patient(start_event):
    timeline = []
    missing_evidence = []
    visited = set()
    pending = [start_event]

    while pending:
        event_name = pending.pop(0)
        if event_name in visited:
            continue
        visited.add(event_name)
        event = find_event(event_name)
        if event is None:
            missing_evidence.append(
                f"Unknown event: {event_name}"
            )
            continue
        timeline.append(event_name)
        if not validate_source(event_name):
            missing_evidence.append(
                f"Missing source for: {event_name}"
            )
        pending.extend(get_next_events(event_name))
    return timeline, missing_evidence

timeline, missing = investigate_patient("Diagnosis")
print("Reconstructed oncology timeline:")
for step, event in enumerate(timeline, start=1):
    print(f"{step}. {event}")
print("\nEvidence issues:")
if missing:
    for issue in missing:
        print("-", issue)
else:
    print("No missing source references found.")